# Анализ жизненного цикла и карточка эксплуатационных моделей

Переработанная версия исходного двухчастного ноутбука.

Периметр формируется напрямую из модельной библиотеки: все эксплуатационные модели категорий A/B, отдельно 50% категории C и отдельно 50% категории D. Категория E и версии без категории исключаются. Для каждой модели выбирается последняя эксплуатационная версия с актуальной категорией значимости.

Ноутбук не читает входные Excel/CSV, не создаёт промежуточные Hive-таблицы и не зависит от результатов предыдущих запусков. Итоговые Excel-файлы сохраняются в текущую рабочую папку Jupyter — при стандартном запуске рядом с ноутбуком.

In [ ]:
import os
import sys

os.environ["SPARK_MAJOR_VERSION"] = "3.5.1"
os.environ["SPARK_HOME"] = "/usr/sdp/current/spark3.5.1-client/"
os.environ["PYSPARK_DRIVER"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
sys.path.insert(0, "/usr/sdp/current/spark3.5.1-client/python/")
sys.path.insert(0, "/usr/sdp/current/spark3.5.1-client/python/lib/py4j-0.10.9.7-src.zip")

from functools import reduce
from pathlib import Path

import pandas as pd
from pyspark import SparkConf, StorageLevel
from pyspark.sql import DataFrame, SparkSession, Window, functions as F

conf = (
    SparkConf()
    .setAppName("model_lifecycle_and_card")
    .setMaster("yarn")
    .set("spark.executor.cores", "2")
    .set("spark.executor.memory", "6g")
    .set("spark.executor.memoryOverhead", "1g")
    .set("spark.driver.memory", "6g")
    .set("spark.driver.maxResultSize", "4g")
    .set("spark.shuffle.service.enabled", "true")
    .set("spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive", "true")
    .set("spark.dynamicAllocation.enabled", "true")
    .set("spark.dynamicAllocation.executorIdleTimeout", "120s")
    .set("spark.dynamicAllocation.cachedExecutorIdleTimeout", "600s")
    .set("spark.dynamicAllocation.initialExecutors", "4")
    .set("spark.dynamicAllocation.maxExecutors", "12")
    .set("spark.dynamicAllocation.shuffleTracking.enabled", "true")
    .set("spark.port.maxRetries", "150")
    .set("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")
    .set("spark.sql.parquet.writeLegacyFormat", "true")
    .set("spark.sql.parquet.compression.codec", "snappy")
    .set("spark.sql.session.timeZone", "Europe/Moscow")
)

spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
SOURCE_DB = "prx_pri_custom_ris_l_library_custom_risk_model_library"
OUTPUT_DIR = Path.cwd().resolve()

SAMPLE_SEED = "model-risk-sample-v1"
HALF_ROUNDING = "ceil"  # ceil или floor для нечётного количества C/D
HISTORY_CUTOFF = None    # например "2025-10-02 00:00:00"; None = вся история
MAX_EXCEL_ROWS = 1_048_575

EXPLOITATION_STATUSES = {
    "IMPLEMENTATION_EXPLOITATION",
    "IMPLEMENTATION_EXPLOITATION_WITHOUT_VALIDATION",
    "EXPLOITATION",
    "ЭКСПЛУАТАЦИЯ",
    "ЭКСПЛУАТАЦИЯ БЕЗ ВАЛИДАЦИИ",
}

TABLES = {
    "model": f"{SOURCE_DB}.t_model",
    "model_ver": f"{SOURCE_DB}.t_model_ver",
    "model_ver_prom": f"{SOURCE_DB}.t_model_ver_prom",
    "prom_implm": f"{SOURCE_DB}.t_prom_implm",
    "pilot_implm": f"{SOURCE_DB}.t_pilot_implm",
    "prom_link": f"{SOURCE_DB}.t_model_ver_prom_x_prom_implm",
    "pilot_link": f"{SOURCE_DB}.t_model_ver_prom_x_pilot_implm",
    "busn_link": f"{SOURCE_DB}.t_model_ver_x_busn_task",
    "busn_task": f"{SOURCE_DB}.t_busn_task",
    "valid": f"{SOURCE_DB}.t_valid",
    "valid_it": f"{SOURCE_DB}.t_valid_it",
    "manual_link": f"{SOURCE_DB}.t_model_ver_x_montrg_manual",
    "manual_result": f"{SOURCE_DB}.t_montrg_manual_rslt",
    "auto_link": f"{SOURCE_DB}.t_model_ver_x_montrg_auto",
    "auto_monitor": f"{SOURCE_DB}.t_montrg_auto",
    "change_log": f"{SOURCE_DB}.t_ent_param_chg",
}

print("Источник:", SOURCE_DB)
print("Результаты:", OUTPUT_DIR)

## Служебные функции и проверка схемы

In [ ]:
def require_table(table_name: str) -> None:
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(f"Не найдена таблица: {table_name}")


def require_columns(df: DataFrame, table_name: str, columns) -> None:
    missing = sorted(set(columns) - set(df.columns))
    if missing:
        raise RuntimeError(f"{table_name}: отсутствуют поля: {', '.join(missing)}")


def ts(value):
    c = F.col(value) if isinstance(value, str) else value
    text = F.trim(c.cast("string"))
    return F.coalesce(
        c.cast("timestamp"),
        F.to_timestamp(text, "dd.MM.yyyy H:mm:ss"),
        F.to_timestamp(text, "dd.MM.yyyy HH:mm:ss"),
        F.to_timestamp(text, "yyyy-MM-dd HH:mm:ss.SSSSSS"),
        F.to_timestamp(text, "yyyy-MM-dd HH:mm:ss"),
        F.to_timestamp(text, "yyyy-MM-dd"),
    )


def norm(value):
    c = F.col(value) if isinstance(value, str) else value
    return F.upper(F.trim(c.cast("string")))


def current_rows(table_name: str, keys, required=()) -> DataFrame:
    # Одна актуальная строка на ключ; удалённая последняя версия не воскрешается.
    require_table(table_name)
    df = spark.table(table_name)
    require_columns(df, table_name, list(keys) + list(required))

    order = []
    if "end_dt" in df.columns:
        end_ts = ts("end_dt")
        is_open = F.col("end_dt").isNull() | (end_ts >= F.current_timestamp()) | (F.year(end_ts) >= 9999)
        order.append(F.when(is_open, 1).otherwise(0).desc())
    if "ctl_datechange" in df.columns:
        order.append(ts("ctl_datechange").desc_nulls_last())
    if "start_dt" in df.columns:
        order.append(ts("start_dt").desc_nulls_last())
    if "end_dt" in df.columns:
        order.append(ts("end_dt").desc_nulls_last())
    order.extend(F.col(key).cast("string").desc_nulls_last() for key in keys)

    window = Window.partitionBy(*keys).orderBy(*order)
    result = df.withColumn("__rn", F.row_number().over(window)).filter("__rn = 1").drop("__rn")
    if "ctl_action" in result.columns:
        result = result.filter(F.coalesce(norm("ctl_action"), F.lit("")) != "D")
    return result


def latest_by(df: DataFrame, keys, order_columns) -> DataFrame:
    order = [ts(c).desc_nulls_last() for c in order_columns if c in df.columns]
    order.extend(F.col(k).cast("string").desc_nulls_last() for k in keys)
    window = Window.partitionBy(*keys).orderBy(*order)
    return df.withColumn("__rn", F.row_number().over(window)).filter("__rn = 1").drop("__rn")


def cap_open_end(value):
    end_ts = ts(value)
    return F.when(
        end_ts.isNull() | (end_ts > F.current_timestamp()) | (F.year(end_ts) >= 9999),
        F.current_timestamp(),
    ).otherwise(end_ts)


def mapping_expr(mapping, value):
    pairs = []
    for key, label in mapping.items():
        pairs.extend([F.lit(key), F.lit(label)])
    return F.coalesce(F.create_map(*pairs)[norm(value)], F.col(value).cast("string"))


for table_name in TABLES.values():
    require_table(table_name)
print("Все таблицы найдены")

## Актуальные срезы источников

In [ ]:
model = current_rows(TABLES["model"], ["model_sid"], [
    "model_name", "model_rsk_flag", "model_rsk_type_name", "model_rsk_sgmnt_name",
    "model_type_name", "model_subtype_name", "model_code", "model_conf_ctgry_name",
    "model_sel_secret_flag", "model_crtn_dttm", "model_stts_name",
])

model_ver = current_rows(TABLES["model_ver"], ["model_ver_sid"], [
    "model_sid", "model_ver_signfcnt_lvl_name", "model_ver_signfcnt_ctgry_code",
    "model_ver_signfcnt_ctgry_dttm", "model_ver_signfcnt_ctgry_descr_txt",
    "model_ver_dev_start_fact_dttm", "model_ver_dev_end_fact_dttm",
    "model_ver_dev_sys_name", "model_ver_data_mart_link_txt", "model_ver_crtn_dttm",
    "model_ver_dev_report_sid", "model_ver_prevalid_report_file_sid",
    "model_ver_prevalid_report_link_sid", "model_ver_prevalid_report_link_txt",
    "model_ver_prevalid_dttm", "model_ver_dev_block_name", "model_ver_dev_dprtmt_name",
    "model_ver_dev_dprtmt_cont_name", "model_ver_stts_name",
])

model_ver_prom = current_rows(TABLES["model_ver_prom"], ["model_ver_prom_sid"], [
    "model_ver_sid", "model_ver_prom_instr_name", "model_ver_prom_sys_name",
    "model_ver_prom_stts_name", "model_ver_prom_crtn_dttm", "model_ver_prom_distr_end_dttm",
    "model_ver_prom_dev_block_name", "model_ver_prom_dev_dprtmt_name",
])

prom_implm = current_rows(TABLES["prom_implm"], ["prom_implm_sid"], [
    "prom_implm_stts_name", "prom_implm_crtn_dttm", "prom_implm_start_fact_dttm",
    "prom_implm_end_fact_dttm", "prom_implm_dprtmt_cont_array", "prom_implm_type_name",
])
pilot_implm = current_rows(TABLES["pilot_implm"], ["pilot_implm_sid"], [
    "pilot_implm_stts_name", "pilot_implm_crtn_dttm", "pilot_implm_start_fact_dttm",
    "pilot_implm_end_fact_dttm", "pilot_implm_dprtmt_cont_array", "pilot_implm_type_name",
])

prom_link = current_rows(TABLES["prom_link"], ["model_ver_prom_sid", "prom_implm_sid"])
pilot_link = current_rows(TABLES["pilot_link"], ["model_ver_prom_sid", "pilot_implm_sid"])
busn_link = current_rows(TABLES["busn_link"], ["model_ver_sid", "busn_task_sid"])
busn_task = current_rows(TABLES["busn_task"], ["busn_task_sid"], [
    "busn_task_name", "busn_task_claim_descr_txt", "busn_task_crtn_dttm",
    "busn_task_start_dttm", "busn_task_end_dttm", "busn_task_employer_block_name",
    "busn_task_employer_dprtmt_name",
])
valid = current_rows(TABLES["valid"], ["valid_sid"], [
    "model_ver_sid", "valid_crtn_dttm", "valid_start_fact_dttm", "valid_end_fact_dttm",
    "valid_report_sid", "valid_dprtmt_name",
])
valid_it = current_rows(TABLES["valid_it"], ["valid_it_sid"], [
    "model_ver_prom_sid", "valid_it_crtn_dttm", "valid_it_start_dttm", "valid_it_end_dttm",
    "valid_it_rslt_name", "valid_it_dprtmt_name",
])
manual_link = current_rows(TABLES["manual_link"], ["model_ver_sid", "montrg_manual_sid"])
manual_result = current_rows(TABLES["manual_result"], ["montrg_manual_rslt_sid"], [
    "montrg_manual_sid", "montrg_manual_rslt_start_dttm", "montrg_manual_rslt_end_dttm",
    "montrg_manual_rslt_report_sid",
])
auto_link = current_rows(TABLES["auto_link"], ["model_ver_sid", "montrg_auto_sid"])
auto_monitor = current_rows(TABLES["auto_monitor"], ["montrg_auto_sid"], [
    "montrg_auto_crtn_dttm", "montrg_auto_start_dttm", "montrg_auto_end_dttm",
    "montrg_auto_dprtmt_name",
])

## Периметр: эксплуатация + A/B + 50% C + 50% D

In [ ]:
prom_active = (
    prom_link.alias("l")
    .join(prom_implm.alias("i"), F.col("l.prom_implm_sid") == F.col("i.prom_implm_sid"), "inner")
    .filter(norm("i.prom_implm_stts_name").isin(*EXPLOITATION_STATUSES))
    .select(
        F.col("l.model_ver_prom_sid").cast("string").alias("model_ver_prom_sid"),
        F.col("i.prom_implm_sid").cast("string").alias("implm_sid"),
        F.lit("PROM").alias("implm_source"),
        F.col("i.prom_implm_stts_name").cast("string").alias("implm_status"),
        F.col("i.prom_implm_crtn_dttm").alias("implm_crtn_dttm"),
        F.col("i.prom_implm_start_fact_dttm").alias("implm_start_fact_dttm"),
        F.col("i.prom_implm_end_fact_dttm").alias("implm_end_fact_dttm"),
        F.col("i.prom_implm_dprtmt_cont_array").cast("string").alias("implm_team"),
        F.col("i.prom_implm_type_name").cast("string").alias("implm_type"),
    )
)

pilot_active = (
    pilot_link.alias("l")
    .join(pilot_implm.alias("i"), F.col("l.pilot_implm_sid") == F.col("i.pilot_implm_sid"), "inner")
    .filter(norm("i.pilot_implm_stts_name").isin(*EXPLOITATION_STATUSES))
    .select(
        F.col("l.model_ver_prom_sid").cast("string").alias("model_ver_prom_sid"),
        F.col("i.pilot_implm_sid").cast("string").alias("implm_sid"),
        F.lit("PILOT").alias("implm_source"),
        F.col("i.pilot_implm_stts_name").cast("string").alias("implm_status"),
        F.col("i.pilot_implm_crtn_dttm").alias("implm_crtn_dttm"),
        F.col("i.pilot_implm_start_fact_dttm").alias("implm_start_fact_dttm"),
        F.col("i.pilot_implm_end_fact_dttm").alias("implm_end_fact_dttm"),
        F.col("i.pilot_implm_dprtmt_cont_array").cast("string").alias("implm_team"),
        F.col("i.pilot_implm_type_name").cast("string").alias("implm_type"),
    )
)

active_implm = prom_active.unionByName(pilot_active).dropDuplicates([
    "model_ver_prom_sid", "implm_sid", "implm_source"
])

operational = (
    model_ver_prom.select(
        F.col("model_ver_prom_sid").cast("string").alias("model_ver_prom_sid"),
        F.col("model_ver_sid").cast("string").alias("model_ver_sid"),
    )
    .join(active_implm, "model_ver_prom_sid", "inner")
)

operational_version_ids = operational.select("model_ver_sid").distinct()
candidates = (
    model_ver.alias("v")
    .join(operational_version_ids.alias("o"), F.col("v.model_ver_sid").cast("string") == F.col("o.model_ver_sid"), "inner")
    .select(
        F.col("v.model_sid").cast("string").alias("model_sid"),
        F.col("v.model_ver_sid").cast("string").alias("model_ver_sid"),
        norm("v.model_ver_signfcnt_ctgry_code").alias("significance"),
        F.col("v.model_ver_signfcnt_ctgry_dttm").alias("significance_dttm"),
        F.col("v.model_ver_crtn_dttm").alias("model_ver_crtn_dttm"),
    )
    .filter(F.col("model_sid").isNotNull() & F.col("significance").isin("A", "B", "C", "D"))
)

latest_version_window = Window.partitionBy("model_sid").orderBy(
    ts("significance_dttm").desc_nulls_last(),
    ts("model_ver_crtn_dttm").desc_nulls_last(),
    F.col("model_ver_sid").desc(),
)
latest_operational = candidates.withColumn(
    "__rn", F.row_number().over(latest_version_window)
).filter("__rn = 1").drop("__rn")

category_window = Window.partitionBy("significance")
sample_window = Window.partitionBy("significance").orderBy(
    F.xxhash64(F.col("model_sid"), F.lit(SAMPLE_SEED)).asc(),
    F.col("model_sid").asc(),
)
ranked = (
    latest_operational
    .withColumn("category_total", F.count("*").over(category_window))
    .withColumn("sample_rank", F.row_number().over(sample_window))
)
if HALF_ROUNDING == "ceil":
    half_count = F.ceil(F.col("category_total") / F.lit(2.0))
elif HALF_ROUNDING == "floor":
    half_count = F.floor(F.col("category_total") / F.lit(2.0))
else:
    raise ValueError("HALF_ROUNDING должен быть 'ceil' или 'floor'")

selected_models = (
    ranked
    .withColumn(
        "keep_count",
        F.when(F.col("significance").isin("C", "D"), half_count).otherwise(F.col("category_total")),
    )
    .filter(F.col("sample_rank") <= F.col("keep_count"))
    .drop("category_total", "sample_rank", "keep_count")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

selection_control = (
    latest_operational.groupBy("significance").count().withColumnRenamed("count", "before_selection")
    .join(
        selected_models.groupBy("significance").count().withColumnRenamed("count", "selected"),
        "significance", "left",
    )
    .orderBy("significance")
)
selection_control.show(truncate=False)

In [ ]:
selected_ids = F.broadcast(selected_models.select("model_sid", "model_ver_sid", "significance"))

selected_implm = (
    selected_ids.select("model_ver_sid")
    .join(operational, "model_ver_sid", "inner")
    .dropDuplicates(["model_ver_sid", "model_ver_prom_sid", "implm_sid", "implm_source"])
)
primary_implm = latest_by(
    selected_implm,
    ["model_ver_sid"],
    ["implm_start_fact_dttm", "implm_crtn_dttm"],
)

selected_busn_links = (
    selected_ids.select("model_ver_sid")
    .join(
        busn_link.select(
            F.col("model_ver_sid").cast("string").alias("model_ver_sid"),
            F.col("busn_task_sid").cast("string").alias("busn_task_sid"),
            "start_dt",
        ),
        "model_ver_sid", "inner",
    )
    .dropDuplicates(["model_ver_sid", "busn_task_sid"])
)

print("Выбрано моделей:", selected_models.count())
print("Связанных действующих внедрений:", selected_implm.count())
print("Связанных бизнес-задач:", selected_busn_links.select("busn_task_sid").distinct().count())

## История статусов бизнес-задач и внедрений

In [ ]:
change_log = spark.table(TABLES["change_log"])
require_columns(change_log, TABLES["change_log"], [
    "ent_sid", "ent_type_name", "ent_param_chg_sid", "ent_param_chg_val",
    "ent_param_chg_prev_val", "ent_param_chg_usr_name", "start_dttm", "end_dttm",
])
if "ctl_action" in change_log.columns:
    change_log = change_log.filter(F.coalesce(norm("ctl_action"), F.lit("")) != "D")
if HISTORY_CUTOFF:
    change_log = change_log.filter(ts("start_dttm") < F.to_timestamp(F.lit(HISTORY_CUTOFF)))

log = change_log.select(
    F.col("ent_sid").cast("string").alias("entity_sid"),
    norm("ent_type_name").alias("entity_type"),
    norm("ent_param_chg_sid").alias("parameter_sid"),
    F.col("ent_param_chg_prev_val").cast("string").alias("previous_value"),
    F.col("ent_param_chg_val").cast("string").alias("new_value"),
    F.col("ent_param_chg_usr_name").cast("string").alias("change_user_name"),
    ts("start_dttm").alias("start_dttm"),
    cap_open_end("end_dttm").alias("end_dttm"),
)

BUSINESS_STATUS_LABELS = {
    "BUSINESS_TASK_BACKLOG": "Ожидает начала (бэклог)",
    "BUSINESS_TASK_DEVELOPMENT": "В работе",
    "BUSINESS_TASK_DECISION_MAKING": "Принятие решения о завершении задачи",
    "BUSINESS_TASK_DONE": "Завершена",
}
IMPLEMENTATION_STATUS_LABELS = {
    "IMPLEMENTATION_FORMATION": "Формирование",
    "IMPLEMENTATION_PREPARATION_PILOT": "Подготовка к пилоту",
    "IMPLEMENTATION_PILOT": "Пилот",
    "IMPLEMENTATION_PILOT_DONE": "Пилот завершён",
    "IMPLEMENTATION_PREPARATION_EXPLOITATION": "Подготовка к эксплуатации",
    "IMPLEMENTATION_WAITING_VALIDATION": "Ожидает валидации",
    "IMPLEMENTATION_WAITING_IT_VALIDATION": "Ожидает IT-валидации",
    "IMPLEMENTATION_EXPLOITATION": "Эксплуатация",
    "IMPLEMENTATION_EXPLOITATION_WITHOUT_VALIDATION": "Эксплуатация без валидации",
    "IMPLEMENTATION_EXPLOITATION_WITHDRAWN": "Выведено из эксплуатации",
    "IMPLEMENTATION_CANCELED": "Отменено",
}

business_ids = selected_busn_links.select(F.col("busn_task_sid").alias("entity_sid")).distinct()
implementation_ids = selected_implm.select(F.col("implm_sid").alias("entity_sid")).distinct()

busn_task_life_stage = (
    business_ids
    .join(log.filter(F.col("parameter_sid") == "BUSINESS_TASK_STATUS"), "entity_sid", "left")
    .select(
        F.col("entity_sid").alias("busn_task_sid"),
        mapping_expr(BUSINESS_STATUS_LABELS, "new_value").alias("life_stage_name"),
        "previous_value", "new_value", "change_user_name", "start_dttm", "end_dttm",
    )
    .orderBy("busn_task_sid", "start_dttm")
)

implm_life_stage = (
    implementation_ids
    .join(log.filter(F.col("parameter_sid") == "IMPLEMENTATION_STATUS"), "entity_sid", "left")
    .select(
        F.col("entity_sid").alias("implm_sid"),
        mapping_expr(IMPLEMENTATION_STATUS_LABELS, "new_value").alias("life_stage_name"),
        "previous_value", "new_value", "change_user_name", "start_dttm", "end_dttm",
    )
    .orderBy("implm_sid", "start_dttm")
)

parameter_changes = (
    selected_ids.select(F.col("model_ver_sid").alias("entity_sid"))
    .join(
        log.filter(F.col("parameter_sid").isin(
            "MODEL_VERSION_IMPORTANCE_UMR", "MODEL_VERSION_MODEL_EFFECT"
        )),
        "entity_sid", "inner",
    )
)
latest_model_parameters = latest_by(
    parameter_changes,
    ["entity_sid", "parameter_sid"],
    ["start_dttm"],
).select(
    F.col("entity_sid").alias("model_ver_sid"),
    "parameter_sid", "previous_value", "new_value", "change_user_name",
    F.col("start_dttm").alias("change_dttm"),
).orderBy("model_ver_sid", "parameter_sid")

## Временная шкала жизненного цикла

In [ ]:
def stage_frame(df: DataFrame, stage_name: str, start_col: str, end_col: str = None) -> DataFrame:
    start_value = ts(start_col)
    end_value = start_value if end_col is None else cap_open_end(end_col)
    normalized_end = F.when(end_value < start_value, start_value).otherwise(end_value)
    return (
        df.select(
            F.col("model_ver_sid").cast("string").alias("model_ver_sid"),
            start_value.alias("start_dt"),
            normalized_end.alias("end_dt"),
            F.lit(stage_name).alias("life_cycle_stage"),
        )
        .filter(F.col("start_dt").isNotNull())
        .groupBy("model_ver_sid", "life_cycle_stage")
        .agg(F.min("start_dt").alias("start_dt"), F.max("end_dt").alias("end_dt"))
        .select("model_ver_sid", "start_dt", "end_dt", "life_cycle_stage")
    )

selected_version_data = (
    selected_ids.select("model_ver_sid")
    .join(
        model_ver.withColumn("model_ver_sid", F.col("model_ver_sid").cast("string")),
        "model_ver_sid", "inner",
    )
)
selected_business = (
    selected_busn_links.alias("l")
    .join(
        busn_task.alias("b"),
        F.col("l.busn_task_sid") == F.col("b.busn_task_sid").cast("string"),
        "inner",
    )
    .select(
        F.col("l.model_ver_sid"),
        F.coalesce(F.col("b.busn_task_start_dttm"), F.col("b.busn_task_crtn_dttm")).alias("stage_start"),
        F.col("b.busn_task_end_dttm").alias("stage_end"),
    )
)
selected_valid = (
    selected_ids.select("model_ver_sid")
    .join(valid.withColumn("model_ver_sid", F.col("model_ver_sid").cast("string")), "model_ver_sid", "inner")
)
selected_auto = (
    selected_ids.select("model_ver_sid")
    .join(auto_link.withColumn("model_ver_sid", F.col("model_ver_sid").cast("string")), "model_ver_sid", "inner")
    .join(auto_monitor, "montrg_auto_sid", "inner")
)
selected_prom = (
    selected_ids.select("model_ver_sid")
    .join(model_ver_prom.withColumn("model_ver_sid", F.col("model_ver_sid").cast("string")), "model_ver_sid", "inner")
)
selected_valid_it = selected_prom.select("model_ver_sid", "model_ver_prom_sid").join(
    valid_it, "model_ver_prom_sid", "inner"
)

lifecycle_parts = [
    stage_frame(selected_business, "Постановка бизнес-задачи", "stage_start", "stage_end"),
    stage_frame(selected_version_data, "Разработка модели", "model_ver_dev_start_fact_dttm", "model_ver_dev_end_fact_dttm"),
    stage_frame(selected_version_data, "Превалидация модели", "model_ver_prevalid_dttm"),
    stage_frame(selected_valid, "Валидация модели", "valid_start_fact_dttm", "valid_end_fact_dttm"),
    stage_frame(selected_auto, "Модель поставлена на автомониторинг", "montrg_auto_start_dttm", "montrg_auto_end_dttm"),
    stage_frame(selected_prom, "Разработка промышленной версии модели", "model_ver_prom_crtn_dttm", "model_ver_prom_distr_end_dttm"),
    stage_frame(selected_valid_it, "IT-валидация", "valid_it_start_dttm", "valid_it_end_dttm"),
    stage_frame(selected_implm, "Внедрение модели", "implm_start_fact_dttm", "implm_end_fact_dttm"),
]

model_lifecycle = (
    reduce(lambda left, right: left.unionByName(right), lifecycle_parts)
    .orderBy("model_ver_sid", "start_dt", "life_cycle_stage")
)
lifecycle_bounds = (
    model_lifecycle.groupBy("model_ver_sid")
    .agg(F.min("start_dt").alias("start_dt"), F.max("end_dt").alias("end_dt"))
    .orderBy("model_ver_sid")
)

## Единая карточка модели

In [ ]:
business_for_card = latest_by(
    selected_busn_links.alias("l")
    .join(
        busn_task.alias("b"),
        F.col("l.busn_task_sid") == F.col("b.busn_task_sid").cast("string"),
        "left",
    )
    .select(
        F.col("l.model_ver_sid"), F.col("l.busn_task_sid"), F.col("l.start_dt").alias("busn_link_start_dt"),
        F.col("b.busn_task_name"), F.col("b.busn_task_claim_descr_txt"),
        F.col("b.busn_task_crtn_dttm"), F.col("b.busn_task_employer_block_name"),
        F.col("b.busn_task_employer_dprtmt_name"),
    ),
    ["model_ver_sid"], ["busn_link_start_dt", "busn_task_crtn_dttm"],
)

manual_joined = (
    selected_ids.select("model_ver_sid").alias("s")
    .join(
        manual_link.alias("l"),
        F.col("s.model_ver_sid") == F.col("l.model_ver_sid").cast("string"),
        "left",
    )
    .join(
        manual_result.alias("r"),
        F.col("l.montrg_manual_sid") == F.col("r.montrg_manual_sid"),
        "left",
    )
    .select(
        F.col("s.model_ver_sid"), F.col("l.montrg_manual_sid"),
        F.col("l.start_dt").alias("manual_link_start_dt"),
        F.col("r.montrg_manual_rslt_start_dttm"), F.col("r.montrg_manual_rslt_end_dttm"),
        F.col("r.montrg_manual_rslt_report_sid"), F.col("r.montrg_manual_rslt_sid"),
    )
)
manual_for_card = latest_by(
    manual_joined,
    ["model_ver_sid"], ["montrg_manual_rslt_start_dttm", "manual_link_start_dt"],
).select(
    "model_ver_sid", "montrg_manual_sid", "montrg_manual_rslt_end_dttm",
    "montrg_manual_rslt_start_dttm", "montrg_manual_rslt_report_sid",
)

auto_joined = (
    selected_ids.select("model_ver_sid").alias("s")
    .join(
        auto_link.alias("l"),
        F.col("s.model_ver_sid") == F.col("l.model_ver_sid").cast("string"),
        "left",
    )
    .join(auto_monitor.alias("a"), F.col("l.montrg_auto_sid") == F.col("a.montrg_auto_sid"), "left")
    .select(
        F.col("s.model_ver_sid"), F.col("l.montrg_auto_sid"),
        F.col("l.start_dt").alias("auto_link_start_dt"),
        F.col("a.montrg_auto_crtn_dttm"), F.col("a.montrg_auto_start_dttm"),
        F.col("a.montrg_auto_end_dttm"), F.col("a.montrg_auto_dprtmt_name"),
    )
)
auto_for_card = latest_by(
    auto_joined,
    ["model_ver_sid"], ["montrg_auto_start_dttm", "montrg_auto_crtn_dttm"],
).select(
    "model_ver_sid", "montrg_auto_sid", "montrg_auto_crtn_dttm",
    "montrg_auto_end_dttm", "montrg_auto_dprtmt_name",
)

valid_for_card = latest_by(
    selected_valid,
    ["model_ver_sid"], ["valid_start_fact_dttm", "valid_crtn_dttm"],
).select(
    "model_ver_sid", "valid_crtn_dttm", "valid_sid", "valid_report_sid", "valid_dprtmt_name",
)

prom_for_card = (
    primary_implm.select("model_ver_sid", "model_ver_prom_sid").alias("i")
    .join(model_ver_prom.alias("p"), F.col("i.model_ver_prom_sid") == F.col("p.model_ver_prom_sid").cast("string"), "left")
    .select(
        F.col("i.model_ver_sid"), F.col("i.model_ver_prom_sid"),
        F.col("p.model_ver_prom_instr_name"), F.col("p.model_ver_prom_sys_name"),
        F.col("p.model_ver_prom_stts_name"), F.col("p.model_ver_prom_crtn_dttm"),
        F.col("p.model_ver_prom_dev_block_name"), F.col("p.model_ver_prom_dev_dprtmt_name"),
    )
)

valid_it_for_card = latest_by(
    prom_for_card.select("model_ver_sid", "model_ver_prom_sid").join(valid_it, "model_ver_prom_sid", "left"),
    ["model_ver_sid"], ["valid_it_start_dttm", "valid_it_crtn_dttm"],
).select(
    "model_ver_sid", "valid_it_end_dttm", "valid_it_start_dttm", "valid_it_crtn_dttm",
    "valid_it_rslt_name", "valid_it_dprtmt_name",
)

card_base = (
    selected_ids.alias("s")
    .join(model_ver.alias("v"), F.col("s.model_ver_sid") == F.col("v.model_ver_sid").cast("string"), "inner")
    .join(model.alias("m"), F.col("s.model_sid") == F.col("m.model_sid").cast("string"), "left")
    .select(
        F.col("s.model_sid").alias("model_sid"),
        F.col("s.model_ver_sid").alias("model_ver_sid"),
        F.col("s.significance").alias("significance"),
        F.col("m.model_name"), F.col("m.model_rsk_flag"), F.col("m.model_rsk_type_name"),
        F.col("m.model_rsk_sgmnt_name"), F.col("m.model_type_name"), F.col("m.model_subtype_name"),
        F.col("m.model_code"), F.col("m.model_conf_ctgry_name"), F.col("m.model_sel_secret_flag"),
        F.col("m.model_stts_name"), F.col("v.model_ver_signfcnt_lvl_name"),
        F.col("v.model_ver_dev_start_fact_dttm"), F.col("v.model_ver_signfcnt_ctgry_descr_txt"),
        F.col("v.model_ver_dev_end_fact_dttm"), F.col("v.model_ver_dev_sys_name"),
        F.col("v.model_ver_data_mart_link_txt"), F.col("v.model_ver_crtn_dttm"),
        F.col("v.model_ver_dev_report_sid"), F.col("v.model_ver_prevalid_report_file_sid"),
        F.col("v.model_ver_prevalid_report_link_sid"), F.col("v.model_ver_prevalid_report_link_txt"),
        F.col("v.model_ver_dev_block_name"), F.col("v.model_ver_dev_dprtmt_name"),
        F.col("v.model_ver_stts_name"),
    )
)

card = (
    card_base
    .join(business_for_card.alias("b"), "model_ver_sid", "left")
    .join(prom_for_card.alias("p"), "model_ver_sid", "left")
    .join(manual_for_card.alias("hm"), "model_ver_sid", "left")
    .join(auto_for_card.alias("am"), "model_ver_sid", "left")
    .join(valid_for_card.alias("val"), "model_ver_sid", "left")
    .join(valid_it_for_card.alias("vit"), "model_ver_sid", "left")
    .select(
        F.col("model_name").alias("MODEL_NAME|Наименование модели"),
        F.col("model_rsk_flag").alias("MODEL_RSK_FLAG|Флаг риск-модели"),
        F.col("model_rsk_type_name").alias("MODEL_RSK_TYPE_NAME|Тип риска модели"),
        F.col("model_rsk_sgmnt_name").alias("MODEL_RSK_SGMNT_NAME|Наименование риск-сегмента модели"),
        F.col("model_type_name").alias("MODEL_TYPE_NAME|Тип модели"),
        F.col("model_subtype_name").alias("MODEL_SUBTYPE_NAME|Подтип модели"),
        F.col("model_code").alias("MODEL_CODE|Код модели"),
        F.col("model_conf_ctgry_name").alias("MODEL_CONF_CTGRY_NAME|Категория конфиденциальности информации о модели"),
        F.col("model_sel_secret_flag").alias("MODEL_SEL_SECRET_FLAG|Флаг коммерческой тайны"),
        F.col("model_sid").alias("MODEL_SID|Идентификатор модели"),
        F.col("model_ver_signfcnt_lvl_name").alias("MODEL_VER_SIGNFCNT_LVL_NAME|Наименование степени значимости версии модели"),
        F.col("model_ver_dev_start_fact_dttm").alias("MODEL_VER_DEV_START_FACT_DTTM|Фактическая дата-время начала разработки версии модели"),
        F.col("significance").alias("MODEL_VER_SIGNFCNT_CTGRY_CODE|Наименование категории значимости версии модели"),
        F.col("model_ver_signfcnt_ctgry_descr_txt").alias("MODEL_VER_SIGNFCNT_CTGRY_DESCR_TXT|Обоснование категории значимости версии модели"),
        F.col("model_ver_dev_end_fact_dttm").alias("MODEL_VER_DEV_END_FACT_DTTM|Фактическая дата-время окончания разработки версии модели"),
        F.col("model_ver_dev_sys_name").alias("MODEL_VER_DEV_SYS_NAME|Система в которой разрабатывалась версия модели"),
        F.col("model_ver_data_mart_link_txt").alias("MODEL_VER_DATA_MART_LINK_TXT|Ссылка на витрину данных для обучения версии модели"),
        F.col("model_ver_crtn_dttm").alias("MODEL_VER_CRTN_DTTM|Дата-время создания версии модели"),
        F.col("model_ver_dev_report_sid").alias("MODEL_VER_DEV_REPORT_SID|Идентификатор отчета о разработке версия модели"),
        F.col("model_ver_prevalid_report_file_sid").alias("MODEL_VER_PREVALID_REPORT_FILE_SID|Идентификатор файла с отчетом о превалидации версии модели"),
        F.col("model_ver_prevalid_report_link_sid").alias("MODEL_VER_PREVALID_REPORT_LINK_SID|Идентификатор ссылки на отчет о превалидации версии модели"),
        F.col("model_ver_prevalid_report_link_txt").alias("MODEL_VER_PREVALID_REPORT_LINK_TXT|Ссылка на отчет о превалидации"),
        F.col("model_ver_dev_block_name").alias("MODEL_VER_DEV_BLOCK_NAME|Наименование блока разработки версии модели"),
        F.col("model_ver_dev_dprtmt_name").alias("MODEL_VER_DEV_DPRTMT_NAME|Наименование подразделения разработки версии модели"),
        F.col("model_ver_sid").alias("MODEL_VER_SID|Идентификатор версии модели"),
        F.col("model_ver_stts_name").alias("MODEL_VER_STTS_NAME|Наименование статуса версии модели"),
        F.col("model_stts_name").alias("MODEL_STTS_NAME|Статус модели"),
        F.col("b.busn_task_sid").alias("BUSN_TASK_SID|Идентификатор бизнес-задачи"),
        F.col("p.model_ver_prom_instr_name").alias("MODEL_VER_PROM_INSTR_NAME|Среда (инструмент) исполнения промышленной версии модели"),
        F.col("p.model_ver_prom_sys_name").alias("MODEL_VER_PROM_SYS_NAME|Система в которой реализована промышленная версия модели"),
        F.col("p.model_ver_prom_sid").alias("MODEL_VER_PROM_SID|Идентификатор промышленной версии модели"),
        F.col("p.model_ver_prom_stts_name").alias("MODEL_VER_PROM_STTS_NAME|Статус промышленной версии модели"),
        F.col("p.model_ver_prom_crtn_dttm").alias("MODEL_VER_PROM_CRTN_DTTM|Дата-время создания промышленной версии модели"),
        F.col("p.model_ver_prom_dev_block_name").alias("MODEL_VER_PROM_DEV_BLOCK_NAME|Блок разработки промышленной версии модели"),
        F.col("p.model_ver_prom_dev_dprtmt_name").alias("MODEL_VER_PROM_DEV_DPRTMT_NAME|Подразделение разработки промышленной версии модели"),
        F.col("hm.montrg_manual_sid").alias("MONTRG_MANUAL_SID|Идентификатор ручного мониторинга"),
        F.col("am.montrg_auto_sid").alias("MONTRG_AUTO_SID|Идентификатор автоматического мониторинга"),
        F.col("val.valid_crtn_dttm").alias("VALID_CRTN_DTTM|Дата-время создания валидации"),
        F.col("val.valid_sid").alias("VALID_SID|Идентификатор валидации"),
        F.col("val.valid_report_sid").alias("VALID_REPORT_SID|Идентификатор отчета о валидации"),
        F.col("val.valid_dprtmt_name").alias("VALID_DPRTMT_NAME|Подразделение, проводящее валидацию"),
        F.col("b.busn_task_name").alias("BUSN_TASK_NAME|Наименование бизнес-задачи"),
        F.col("b.busn_task_claim_descr_txt").alias("BUSN_TASK_CLAIM_DESCR_TXT|Описание требований бизнес-задачи"),
        F.col("b.busn_task_crtn_dttm").alias("BUSN_TASK_CRTN_DTTM|Дата-время создания бизнес-задачи"),
        F.col("b.busn_task_employer_block_name").alias("BUSN_TASK_EMPLOYER_BLOCK_NAME|Блок заказчика бизнес-задачи"),
        F.col("b.busn_task_employer_dprtmt_name").alias("BUSN_TASK_EMPLOYER_DPRTMT_NAME|Наименование подразделения заказчика бизнес-задачи"),
        F.col("hm.montrg_manual_rslt_end_dttm").alias("MONTRG_MANUAL_RSLT_END_DTTM|Дата-время окончания процесса ручного мониторинга в рамках которого получен текущий результат"),
        F.col("hm.montrg_manual_rslt_start_dttm").alias("MONTRG_MANUAL_RSLT_START_DTTM|Дата-время начала процесса ручного мониторинга в рамках которого получен текущий результат"),
        F.col("hm.montrg_manual_rslt_report_sid").alias("MONTRG_MANUAL_RSLT_REPORT_SID|Идентификатор отчёта о результате ручного мониторинга"),
        F.col("am.montrg_auto_crtn_dttm").alias("MONTRG_AUTO_CRTN_DTTM|Дата-время создания автоматического мониторинга"),
        F.col("am.montrg_auto_end_dttm").alias("MONTRG_AUTO_END_DTTM|Дата-время окончания мониторинга"),
        F.col("am.montrg_auto_dprtmt_name").alias("MONTRG_AUTO_DPRTMT_NAME|Наименование подразделения проводящего автоматического мониторинга"),
        F.col("vit.valid_it_end_dttm").alias("VALID_IT_END_DTTM|Дата-время окончания ИТ-валидации"),
        F.col("vit.valid_it_start_dttm").alias("VALID_IT_START_DTTM|Дата-время начала ИТ-валидации"),
        F.col("vit.valid_it_crtn_dttm").alias("VALID_IT_CRTN_DTTM|Дата-время создания ИТ-валидации"),
        F.col("vit.valid_it_rslt_name").alias("VALID_IT_RSLT_NAME|Результат ИТ-валидации"),
        F.col("vit.valid_it_dprtmt_name").alias("VALID_IT_DPRTMT_NAME|Подразделение проводящее ИТ-валидацию"),
    )
    .orderBy("MODEL_VER_SIGNFCNT_CTGRY_CODE|Наименование категории значимости версии модели", "MODEL_SID|Идентификатор модели")
    .persist(StorageLevel.MEMORY_AND_DISK)
)

## Контроль качества и выгрузка

In [ ]:
card_count = card.count()
selected_count = selected_models.count()
duplicate_count = card.groupBy(
    "MODEL_VER_SID|Идентификатор версии модели"
).count().filter("count > 1").count()

if card_count != selected_count:
    raise RuntimeError(f"Карточка содержит {card_count} строк при {selected_count} выбранных моделях")
if duplicate_count:
    raise RuntimeError(f"В карточке найдены дубликаты версий: {duplicate_count}")

quoted = lambda name: F.col(f"`{name}`")
missing_agg = card.agg(*[
    F.sum(
        F.when(
            quoted(name).isNull() | (F.trim(quoted(name).cast("string")) == ""),
            1,
        ).otherwise(0)
    ).alias(name)
    for name in card.columns
]).first().asDict()

missingness_pdf = pd.DataFrame({
    "Key": card.columns,
    "Value": [int(missing_agg[name]) for name in card.columns],
    "MissingPct": [round(100.0 * int(missing_agg[name]) / card_count, 2) if card_count else 0.0 for name in card.columns],
}).sort_values(["Value", "Key"], ascending=[False, True])

print(f"Карточка: {card_count} строк, дубликатов нет")
selection_control.show(truncate=False)

In [ ]:
def to_pandas_limited(df: DataFrame, name: str) -> pd.DataFrame:
    result = df.limit(MAX_EXCEL_ROWS + 1).toPandas()
    if len(result) > MAX_EXCEL_ROWS:
        raise RuntimeError(f"{name}: превышен лимит строк одного листа Excel")
    return result


def save_xlsx(data, path: Path, sheet_name="data"):
    if isinstance(data, DataFrame):
        data = to_pandas_limited(data, path.name)
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        data.to_excel(writer, index=False, sheet_name=sheet_name)
        sheet = writer.book[sheet_name]
        sheet.freeze_panes = "A2"
        sheet.auto_filter.ref = sheet.dimensions
        for cells in sheet.columns:
            values = ["" if cell.value is None else str(cell.value) for cell in cells[:2000]]
            width = min(max(max((len(value) for value in values), default=0) + 2, 12), 60)
            sheet.column_dimensions[cells[0].column_letter].width = width


outputs = {
    "busn_task_life_stage.xlsx": busn_task_life_stage,
    "df_implm_life_stage.xlsx": implm_life_stage,
    "model_with_latest_importance_values.xlsx": latest_model_parameters,
    "data_to_process_mining.xlsx": model_lifecycle,
    "data_life_stage.xlsx": lifecycle_bounds,
    "output.xlsx": card,
    "example.xlsx": missingness_pdf,
    "selection_control.xlsx": selection_control,
}

for filename, data in outputs.items():
    save_xlsx(data, OUTPUT_DIR / filename)
    print("Сохранён:", OUTPUT_DIR / filename)

## Что изменено относительно исходника

- удалены все ручные входы `temp_busn_task_sid.xls`, `implm_sid.xls`, `all_model.xls`, `data.csv`;
- устранены промежуточные Hive-таблицы `DDD_*` и последовательные чтения/перезаписи;
- один источник данных и единый актуальный срез записей;
- исправлены CTE, импорты, неверные имена файлов и зависимость от старых таблиц;
- `end_dt` рассчитывается по фактическому окончанию, а не через `MAX(start_dttm)`;
- выбор связанных сущностей детерминирован, одна итоговая строка на версию;
- категории E исключаются до всех тяжёлых объединений;
- файл старого сравнения переименован: два разных параметра больше не выдаются за старое и новое значение одного поля;
- добавлен контроль размеров, числа строк, дубликатов и доли пропусков.

In [ ]:
selected_models.unpersist()
card.unpersist()
# spark.stop()  # включите, если после ноутбука Spark-сессия больше не нужна